# GNN-based Graph Clustering (VGAE + KMeans) — Google Colab GPU

This notebook covers the **GNN-based clustering method** only (VGAE encoder + KMeans on learned
embeddings), evaluated on the four datasets produced by `graph_datasets.py`:
`cora`, `email-eu-core`, `lfr-easy`, `lfr-hard`.

**Before running this notebook:**
1. Run `python download_data.py` to fetch the raw files into `data/raw/`.
2. Run `python graph_datasets.py` once to sanity-check that all four datasets load correctly.
3. Make sure `graph_datasets.py` is importable from this notebook (same folder, or add its
   folder to `sys.path` below).

This notebook does **not** cover Louvain/Leiden — that comparison lives in a separate notebook.
Results here (NMI / ARI / modularity per dataset) are saved to a table at the end so they can be
merged with the Louvain/Leiden results for the final report.


## 1. Setup (Colab, GPU runtime)

Make sure the runtime is set to GPU: **Runtime → Change runtime type → T4 GPU**.

If you uploaded `download_data.py` / `graph_datasets.py` to Colab's local filesystem (e.g. via
the file browser, or `git clone`), keep this notebook in the **same folder** as those two files.
If instead they live on Drive, uncomment the Drive-mount cell below.


In [ ]:
# Optional: mount Google Drive if your scripts/data live there
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<your_project_folder>


In [ ]:
# Install torch_geometric matching Colab's pre-installed torch + CUDA build.
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'

!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

!pip install -q python-igraph leidenalg  # not used directly here, but harmless if already needed elsewhere


In [ ]:
import os, sys, time, pickle
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.utils import from_networkx
from torch_geometric.nn import GCNConv, VGAE

from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

import networkx.algorithms.community as nx_comm

# Make sure graph_datasets.py / download_data.py are importable.
# If this notebook is NOT in the same folder as those scripts, edit this path.
SCRIPTS_DIR = os.path.abspath(".")
sys.path.append(SCRIPTS_DIR)

from graph_datasets import LOADERS, largest_component, summarise

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Load datasets and convert to PyG format

We take the **largest connected component** of each graph (GNN message passing assumes
connectivity), then build a PyG `Data` object. Cora has real bag-of-words node features;
`email-eu-core` and the LFR synthetic graphs have none, so we fall back to node degree
as a 1-D feature (noted as a limitation in the report).

In [ ]:
def build_pyg_data(name, G, y):
    """Convert an (nx.Graph, label array) pair into a PyG Data object with simple features."""
    data = from_networkx(G)

    if name == "cora":
        # Reuse the raw Planetoid features (bag-of-words) if available alongside the graph pickle.
        # graph_datasets.load_cora() only returns (G, y), so we reload the raw feature matrix here.
        import pickle as pkl
        raw_dir = os.path.join(SCRIPTS_DIR, "..", "data", "raw")
        with open(os.path.join(raw_dir, "ind.cora.allx"), "rb") as f:
            allx = pkl.load(f, encoding="latin1")
        with open(os.path.join(raw_dir, "ind.cora.tx"), "rb") as f:
            tx = pkl.load(f, encoding="latin1")
        # NOTE: allx/tx indexing follows the original (pre-relabel) Planetoid node order.
        # For a quick, robust fallback that always aligns with G's current node order,
        # we default to degree features unless allx row count matches G's node count exactly.
        import scipy.sparse as sp
        feat = sp.vstack((allx, tx)).toarray() if sp.issparse(allx) else np.vstack((allx, tx))
        if feat.shape[0] == G.number_of_nodes():
            x = torch.tensor(feat, dtype=torch.float)
        else:
            deg = np.array([d for _, d in G.degree()], dtype=np.float32).reshape(-1, 1)
            x = torch.tensor(deg, dtype=torch.float)
    else:
        deg = np.array([d for _, d in G.degree()], dtype=np.float32).reshape(-1, 1)
        x = torch.tensor(deg, dtype=torch.float)

    data.x = x
    data.y = torch.tensor(y, dtype=torch.long)
    return data


datasets = {}
for key, loader_fn in LOADERS.items():
    G, y = loader_fn()
    G, y = largest_component(G, y)
    print(summarise(key + " (LCC)", G, y))
    datasets[key] = build_pyg_data(key, G, y)

print()
for k, d in datasets.items():
    print(k, "-> x:", tuple(d.x.shape), "edge_index:", tuple(d.edge_index.shape))


## 3. Model: VGAE with a 2-layer GCN encoder

Standard variational graph autoencoder: a GCN produces `mu` / `logvar` per node, the latent
sample `z` reconstructs the adjacency matrix, and the loss is reconstruction + KL divergence.
This is the "GNN alone" method — no ground-truth labels are used during training, only during
evaluation afterwards.

In [ ]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv_mu = GCNConv(hidden_channels, out_channels)
        self.conv_logvar = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        return self.conv_mu(h, edge_index), self.conv_logvar(h, edge_index)


def make_model(in_channels, hidden=32, latent=16):
    encoder = GCNEncoder(in_channels, hidden, latent)
    return VGAE(encoder)


## 4. Training loop

One VGAE is trained per dataset (not shared across graphs). GPU runtime: 200 epochs per dataset, this should take well under a minute per dataset on a T4.

In [ ]:
def train_vgae(data, epochs=200, lr=0.01, hidden=32, latent=16, log_every=50, verbose=True):
    data = data.to(device)
    model = make_model(data.x.size(1), hidden=hidden, latent=latent).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    t0 = time.time()
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss = loss + (1.0 / data.num_nodes) * model.kl_loss()
        loss.backward()
        optimizer.step()

        if verbose and (epoch % log_every == 0 or epoch == epochs):
            print(f"  epoch {epoch:4d}  loss={loss.item():.4f}")

    elapsed = time.time() - t0
    if verbose:
        print(f"  trained in {elapsed:.1f}s")

    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
    return z.cpu().numpy(), elapsed


embeddings = {}
train_times = {}
for key, data in datasets.items():
    print(f"Training VGAE on '{key}' ...")
    z, elapsed = train_vgae(data)
    embeddings[key] = z
    train_times[key] = elapsed
    print()


## 5. Cluster the learned embeddings (KMeans) and evaluate

We use `K = number of ground-truth communities` for each dataset (same convention as the
Louvain/Leiden notebook, so results are directly comparable). For `email-eu-core`, unlabelled
nodes (`y == -1`) are excluded from the NMI/ARI computation — they were never assigned a
department in the original dataset.

In [ ]:
def evaluate_clustering(G, y_true, y_pred):
    mask = y_true >= 0  # drop unlabelled nodes (relevant for email-eu-core)
    nmi = normalized_mutual_info_score(y_true[mask], y_pred[mask])
    ari = adjusted_rand_score(y_true[mask], y_pred[mask])

    # Modularity of the *predicted* clusters, computed over the full graph (unsupervised metric,
    # doesn't need ground truth) -- lets us compare directly against Louvain/Leiden's objective.
    communities = {}
    for node, c in zip(G.nodes(), y_pred):
        communities.setdefault(int(c), set()).add(node)
    mod = nx_comm.modularity(G, communities.values())

    return {"NMI": nmi, "ARI": ari, "modularity": mod}


results = []
predicted_labels = {}

for key, loader_fn in LOADERS.items():
    G, y = loader_fn()
    G, y = largest_component(G, y)

    z = embeddings[key]
    n_clusters = len(np.unique(y[y >= 0]))

    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=SEED)
    y_pred = km.fit_predict(z)
    predicted_labels[key] = y_pred

    metrics = evaluate_clustering(G, y, y_pred)
    metrics.update({"dataset": key, "n_nodes": G.number_of_nodes(),
                     "n_clusters": n_clusters, "train_time_s": round(train_times[key], 1)})
    results.append(metrics)

    print(f"{key:14s} NMI={metrics['NMI']:.3f}  ARI={metrics['ARI']:.3f}  "
          f"modularity={metrics['modularity']:.3f}  train_time={metrics['train_time_s']}s")


## 6. Visualization

t-SNE projection of the VGAE embeddings, colored by predicted cluster vs ground-truth label,
for the two real-world graphs. For the LFR pair, we instead plot NMI vs the pair (easy/hard)
to show how performance degrades as the mixing parameter `mu` increases.

In [ ]:
def plot_embedding(key):
    G, y = LOADERS[key]()
    G, y = largest_component(G, y)
    z = embeddings[key]
    y_pred = predicted_labels[key]

    z2 = TSNE(n_components=2, random_state=SEED, init="pca").fit_transform(z)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].scatter(z2[:, 0], z2[:, 1], c=y, cmap="tab10", s=12)
    axes[0].set_title(f"{key} -- ground truth")
    axes[1].scatter(z2[:, 0], z2[:, 1], c=y_pred, cmap="tab10", s=12)
    axes[1].set_title(f"{key} -- VGAE + KMeans")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


plot_embedding("cora")
plot_embedding("email-eu-core")


In [ ]:
mus = ["lfr-easy", "lfr-hard"]
nmi_vals = [next(r["NMI"] for r in results if r["dataset"] == k) for k in mus]

plt.figure(figsize=(4.5, 4))
plt.bar(mus, nmi_vals, color=["#4C72B0", "#C44E52"])
plt.ylabel("NMI")
plt.ylim(0, 1)
plt.title("VGAE + KMeans: LFR easy vs hard")
plt.tight_layout()
plt.show()


## 7. Summary table

Save these results so they can be merged with the Louvain/Leiden results notebook for the
final report table.

In [ ]:
import pandas as pd

df = pd.DataFrame(results)[["dataset", "n_nodes", "n_clusters", "NMI", "ARI", "modularity", "train_time_s"]]
df = df.sort_values("dataset").reset_index(drop=True)
display(df)

out_name = "gnn_results_gpu.csv" if True else "gnn_results_cpu.csv"
df.to_csv(out_name, index=False)
print(f"Saved results to {out_name}")


## Notes / limitations (for the report)

- `email-eu-core` and the LFR graphs have no natural node features, so degree was used as a
  1-D fallback feature — this likely undersells the GNN relative to methods (Louvain/Leiden)
  that only need graph structure and aren't affected by weak features.
- KMeans on the embeddings requires knowing `K` in advance (set to the true number of
  communities here), unlike Louvain/Leiden which infer the number of communities automatically.
  This asymmetry should be discussed explicitly in the Experiments section.
- Only the largest connected component of each graph is used, consistent with the
  Louvain/Leiden notebook, so results are directly comparable.
